# Practice Lab: Caching Strategies for LLM Apps (Lesson 12.4)

Module 12 . 4 exercises . Hit rates + thresholds + vector benchmarks + invalidation


# Lesson 12.4: Caching Strategies for LLM Apps

4 exercises: 2E + 1M + 1C

All exercises run **locally** (pure Python simulations).


In [ ]:
import numpy as np
import random, time, hashlib
np.random.seed(42)
random.seed(42)


---
## Exercise 1 (Easy): Hit Rate Experiment


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random, time, hashlib, numpy as np
random.seed(42); np.random.seed(42)

class DualCache:
    def __init__(self,th=0.85):
        self.exact={}; self.sem=[]; self.th=th
        self.stats={"exact":0,"semantic":0,"miss":0}; self.saved=0.0; self.cost=0.003
    def _h(self,q): return hashlib.md5(q.lower().strip().encode()).hexdigest()
    def _emb(self,t):
        v=np.zeros(32)
        for w in t.lower().split(): v[hash(w)%32]+=1.0
        n=np.linalg.norm(v); return v/n if n>0 else v
    def query(self,prompt):
        h=self._h(prompt)
        if h in self.exact: self.stats["exact"]+=1; self.saved+=self.cost; return "EXACT",0.1
        qe=self._emb(prompt)
        for emb,_ in self.sem:
            if float(np.dot(qe,emb))>=self.th: self.stats["semantic"]+=1; self.saved+=self.cost; return "SEM",5.0
        self.stats["miss"]+=1; self.exact[h]=f"R:{prompt[:20]}"; self.sem.append((self._emb(prompt),self.exact[h]))
        return "MISS",800.0

topics=["course price","refund","prereqs","schedule","cert","modules","capstone","instructor","duration","GenAI vs Python"]
unique=[f"Tell me about {topics[i%10]} v{i}" for i in range(50)]
repeats=[random.choice(unique) for _ in range(50)]
all_q=unique+repeats; random.shuffle(all_q)

cache=DualCache(0.85); lats=[]
for q in all_q:
    _,lat=cache.query(q); lats.append(lat)

total=sum(cache.stats.values()); hit_pct=(cache.stats["exact"]+cache.stats["semantic"])/total*100
print("Hit Rate Experiment (100 queries):")
print(f"  Exact: {cache.stats['exact']} | Semantic: {cache.stats['semantic']} | Miss: {cache.stats['miss']}")
print(f"  Hit rate: {hit_pct:.0f}%")
avg_lat=sum(lats)/len(lats)
print(f"  Latency: {avg_lat:.0f}ms avg (vs 800ms no cache) = {800/avg_lat:.1f}x speedup")
no_cost=total*cache.cost; actual=cache.stats["miss"]*cache.cost
print(f"  Cost: ${actual:.3f} vs ${no_cost:.3f} = ${cache.saved:.3f} saved ({cache.saved/no_cost*100:.0f}%)")


</details>


---
## Exercise 2 (Easy): Threshold Tuning


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np; np.random.seed(42)

def emb(t):
    v=np.zeros(32)
    for w in t.lower().split(): v[hash(w)%32]+=1.0
    n=np.linalg.norm(v); return v/n if n>0 else v
def cos(a,b): return float(np.dot(a,b))

pairs=[("Course price?","How much does it cost?",True),("Prerequisites?","What do I need?",True),
    ("How long?","Course duration?",True),("Refund policy?","Get money back?",True),
    ("Classes start?","Next batch?",True),("Certificate?","Get a cert?",True),
    ("Modules?","Topics covered?",True),("Instructor?","Who teaches?",True),
    ("Capstone?","Final project?",True),("Pay EMI?","Installment?",True),
    ("Schedule?","When are classes?",True),("Online?","Attend from home?",True),
    ("Tools learned?","Technologies?",True),("Assignments?","Homework?",True),
    ("Enroll?","Registration?",True),
    ("Course price?","Design RAG",False),("How long?","Write poem",False),
    ("Refund?","Explain transformers",False),("Classes?","Backpropagation?",False),
    ("Certificate?","Deploy Cloud Run",False),("Modules?","PostgreSQL vs Mongo",False),
    ("Instructor?","Build chatbot",False),("EMI?","Attention mechanism",False),
    ("Schedule?","Binary search",False),("Enroll?","Gradient descent",False),
    ("Online?","Docker?",False),("Tools?","TCP/IP?",False),
    ("Assignments?","Neural network?",False),("Price?","Kubernetes",False),("Duration?","Recursion?",False)]

print("Threshold Tuning (30 pairs):")
print(f"  {'Thresh':>7} {'TP':>4} {'FP':>4} {'TN':>4} {'FN':>4} {'Prec':>6} {'Rec':>6} {'F1':>6}")
best_f1=0; best_th=0
for th in [0.80,0.85,0.90,0.95]:
    tp=fp=tn=fn=0
    for q1,q2,same in pairs:
        match=cos(emb(q1),emb(q2))>=th
        if same and match: tp+=1
        elif not same and match: fp+=1
        elif not same and not match: tn+=1
        else: fn+=1
    prec=tp/max(tp+fp,1); rec=tp/max(tp+fn,1); f1=2*prec*rec/max(prec+rec,0.001)
    if f1>best_f1: best_f1=f1; best_th=th
    print(f"  {th:>7.2f} {tp:>4} {fp:>4} {tn:>4} {fn:>4} {prec:>5.0%} {rec:>5.0%} {f1:>5.0%}")
print(f"\n  Best: {best_th} (F1={best_f1:.0%})")


</details>


---
## Exercise 3 (Medium): Vector Search Benchmark


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np, time; np.random.seed(42)

class VecCache:
    def __init__(self,dim=64): self.vecs=[]; self.dim=dim
    def add(self,emb): self.vecs.append(emb)
    def search(self,q,th=0.85):
        if not self.vecs: return None,0
        mat=np.array(self.vecs); norms=np.linalg.norm(mat,axis=1,keepdims=True)
        sims=(mat/(norms+1e-8))@(q/(np.linalg.norm(q)+1e-8))
        best=float(np.max(sims)); return (best>=th),best

print("Vector Search Benchmark:")
print(f"  {'N':>10} {'p50':>8} {'p95':>8} {'p99':>8} {'Status':>10}")
for n in [100,1000,10000]:
    c=VecCache(64)
    for _ in range(n): c.add(np.random.randn(64))
    lats=[]
    for _ in range(100):
        q=np.random.randn(64); t=time.perf_counter(); c.search(q); lats.append((time.perf_counter()-t)*1000)
    p50,p95,p99=np.percentile(lats,50),np.percentile(lats,95),np.percentile(lats,99)
    st="OK" if p99<10 else "SLOW" if p99<50 else "TOO SLOW"
    print(f"  {n:>10,} {p50:>7.2f}ms {p95:>7.2f}ms {p99:>7.2f}ms {st:>10}")
print(f"\n  <1K: brute-force OK | 1K-10K: borderline | >10K: need HNSW")


</details>


---
## Exercise 4 (Challenge): Cache Invalidation


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import hashlib, time

class TagCache:
    def __init__(self,ttl=3600): self.store={}; self.tags={}; self.ttl=ttl; self.stats={"hit":0,"miss":0,"inv":0}
    def _k(self,p): return hashlib.md5(p.lower().strip().encode()).hexdigest()[:16]
    def get(self,p):
        k=self._k(p); e=self.store.get(k)
        if e and e["exp"]>time.time(): self.stats["hit"]+=1; return e["r"]
        self.stats["miss"]+=1; return None
    def put(self,p,r,tags=None):
        k=self._k(p); self.store[k]={"r":r,"tags":tags or [],"exp":time.time()+self.ttl}
        for t in (tags or []):
            self.tags.setdefault(t,set()).add(k)
    def invalidate(self,tag):
        keys=self.tags.pop(tag,set()).copy(); c=0
        for k in keys:
            if k in self.store: del self.store[k]; c+=1
        self.stats["inv"]+=c; return c

c=TagCache()
for p,r,t in [("Course price?","14999",["pricing"]),("EMI?","5499/mo",["pricing"]),("Discounts?","20% off",["pricing"]),
    ("Bundle?","19999",["pricing"]),("Schedule?","Mon-Fri 7PM",["schedule"]),("Next batch?","Jul 2026",["schedule"]),
    ("Weekend?","Sat-Sun 10AM",["schedule"]),("Holidays?","No national holidays",["schedule"]),
    ("Refund?","7d full",["policy"]),("Transfer?","Next batch",["policy"]),("Attendance?","80% min",["policy"]),
    ("What is RAG?","Retrieval Aug Gen",["general"]),("Agent?","LLM+tools",["general"]),
    ("Prereqs?","Python basics",["general"]),("Certificate?","Completion cert",["general"])]:
    c.put(p,r,t)

print("Cache Invalidation:")
print(f"  Before (cache={len(c.store)}):")
for q,tag in [("Course price?","pricing"),("Schedule?","schedule"),("Refund?","policy"),("What is RAG?","general")]:
    print(f"    [{tag:<9}] {q:<18} -> {'HIT' if c.get(q) else 'MISS'}")

print(f"\n  EVENT: Price changed to 12999!")
removed=c.invalidate("pricing")
print(f"  Invalidated {removed} pricing entries")

print(f"\n  After (cache={len(c.store)}):")
for q,tag in [("Course price?","pricing"),("Schedule?","schedule"),("Refund?","policy"),("What is RAG?","general")]:
    hit=c.get(q); print(f"    [{tag:<9}] {q:<18} -> {'HIT' if hit else 'MISS (cleared)'}")
print(f"\n  Stats: {c.stats}")


</details>
